# COGS 185 PROJECT PROPOSAL

## Title

Comparing Supervised, Self-Supervised, and Recurrent Representation Learning for Structured Spatial Prediction Under Partial Observability

## Abstract

This project investigates how different representation learning paradigms-supervised geometric regression, self-supervised contrastive learning, and predictive modeling-affect structured sequential prediction under partial observability. Using a procedurally generated pixel-based spatial environment, we compare feedforward and recurrent architectures (CNN vs. CNN+LSTM) under multiple training regimes. Representation quality is evaluated using linear probing, transfer performance, sample efficiency, trajectory prediction accuracy, and generalization to unseen layouts. We study whether self-supervised pretraining improves structured sequence modeling and memory-dependent prediction compared to end-to-end supervised learning. This work provides a systematic empirical comparison of advanced representation learning methods in a controlled synthetic environment and contributes insight into how representation learning interacts with recurrent modeling under partial observability.

## Hypotheses

H1: Self-supervised contrastive pretraining improves structured sequence prediction accuracy under partial observability compared to purely supervised training.

H2: Recurrent architectures (CNN+LSTM) outperform feedforward models in environments with delayed rewards and moving obstacles.

H3: Self-supervised representations provide greater sample efficiency in low-data regimes.

## Method Overview

### Environment Formulation

We modify the existing gridworld to enforce partial observability and structured sequence dependence:

- Agent observes only a $5\times5$ egocentric crop.
- Full world state is hidden.
- Add $1$-$2$ moving obstacles with deterministic motion patterns.
- Reward is delayed until goal reached.
- Episode length capped at $T = 40$-$60$ steps.

This transforms the task into:

- A structured sequence prediction problem.
- A memory-dependent modeling problem.
- A partially observable environment requiring latent state inference.

We define the environment as a partially observable Markov decision process (POMDP):

$$
\mathcal{E} = (\mathcal{S}, \mathcal{A}, \mathcal{O}, T, R, \Omega, \gamma)
$$

$$
\gamma \in (0,1)
$$

$$
\gamma = 0.99
$$

The environment is Markovian in full state space but partially observable through observation function $f$.

where:

#### 1. State Space $\mathcal{S}$

- Each state is defined as:

$$
s_t = (p_t^{agent}, p^{goal}, \{p_t^{obs,i}\}_{i=1}^k)
$$

- $p_t^{agent} \in \mathbb{Z}^2$ is agent position on an $N \times N$ grid ($N \in \{8,10\}$).
- $p^{goal} \in \mathbb{Z}^2$ is fixed goal location for episode.
- $p_t^{obs,i} \in \mathbb{Z}^2$ are obstacle positions.
- $k \in \{1,2\}$ controls obstacle count.
- State space size is combinatorial over valid grid configurations.

#### 2. Action Space $\mathcal{A}$

- Discrete movement:

$$
\mathcal{A} = \{\text{up}, \text{down}, \text{left}, \text{right}, \text{stay}\}
$$

#### 3. Observation Function $\Omega$

- Agent does not observe full state.
- Observation:

$$
o_t = f(s_t)
$$

- where $f$ crops a $5\times5$ egocentric window centered at $p_t^{agent}$.
- Thus:

$$
o_t \in \mathbb{R}^{5 \times 5 \times C}
$$

- $C = 3$ (RGB encoding)
- Or binary channel encoding (agent, obstacle, goal layers)
- This induces partial observability.

#### 4. Transition Function $T(s_{t+1} \mid s_t, a_t)$

- Agent transition:

$$
p_{t+1}^{agent} = p_t^{agent} + \Delta(a_t)
$$

- with boundary constraints.
- Obstacle transitions:
- Each obstacle follows deterministic motion:

$$
p_{t+1}^{obs,i} = p_t^{obs,i} + d_i
$$

- where direction $d_i \in \{(1,0), (-1,0), (0,1), (0,-1)\}$.
- If boundary reached, obstacle reverses direction.

#### 5. Reward Function $R(s_t, a_t)$

$$
R_t =
\begin{cases}
+1 & \text{if } p_t^{agent} = p^{goal} \\
-0.01 & \text{otherwise}
\end{cases}
$$

- Optional collision penalty:

$$
-0.1 \text{ if collision with obstacle}
$$

- Reward is delayed until goal.

#### 6. Episode Termination

- Episode ends if:
  - Goal reached.
  - $t = T_{max}$, where $T_{max} \in [40,60]$.

#### 7. Dataset Generation Protocol

- Episodes generated by:
  - Sampling goal uniformly from grid excluding agent start.
  - Sampling obstacle positions uniformly.
  - Sampling obstacle directions uniformly.
  - Agent initial position sampled uniformly.
- Number of episodes:
  - $\geq 10,000$ training episodes.
  - $\geq 2,000$ validation episodes.
  - $\geq 2,000$ test episodes.

#### 8. Train/Test Layout Split

- Two regimes:
  - 1. IID split (same distribution).
  - 2. OOD generalization:
    - Test layouts use unseen obstacle configurations.
    - Possibly larger grid (e.g., train on $8\times8$, test on $10\times10$).

#### 9. Random Seed Control

- All experiments will report:
  - Fixed seed experiments (3 runs).
  - Mean $\pm$ standard deviation.
- This ensures reproducibility.

---

## Representation Learning Methods

All methods use the same CNN backbone to ensure fair comparison.

### CNN Backbone (CPU Configuration)

- Input: $5\times5$ or $9\times9$ RGB patch
- 3 convolution layers:
- Conv(32 filters, $3\times3$)
- Conv(64 filters, $3\times3$)
- Conv(64 filters, $3\times3$)
- ReLU activations
- Optional BatchNorm
- Global average pooling
- Latent embedding size: 64 or 128

This keeps parameter count under $\sim$500k parameters, CPU-feasible.

### Optional GPU Extension

If GPU resources are used:

- Increase embedding dimension to 256
- Use 5 convolution layers
- Add residual connections
- Larger batch sizes (128-256 for contrastive)

This scaling will only affect capacity, not methodology.

### Representation Methods

#### 1. Supervised Geometric Baseline

- Predict:
  - Relative target direction ($\sin \theta$, $\cos \theta$)
- Loss:
  - MSE
- Embedding size:
  - 64 (CPU) / 128-256 (GPU optional)

#### 2. Rotation Prediction (Self-Supervised)

- Predict:
  - Rotation class $\in \{0^\circ, 90^\circ, 180^\circ, 270^\circ\}$
- Loss:
  - Cross-entropy
- Projection head:
  - No projection head required.
- Notes:
  - CPU-friendly by design.

#### 3. Contrastive Learning (SimCLR-lite)

- Setup:
  - For each input frame $x$, two stochastic augmentations are sampled:

$$
x_i = a_i(x), \quad a_i \sim \mathcal{A}
$$

- Augmentation distribution $\mathcal{A}$:
  - Random crop with scale $\in [0.8, 1.0]$
  - Gaussian noise with $\sigma \in [0, 0.05]$
  - Brightness jitter within $\pm 20\%$
- Objective:
  - NT-Xent loss applied to paired augmented views within a batch.
- Projection Head (CPU Configuration):
  - MLP: 64 $\to$ 64 $\to$ 32
  - ReLU between layers

A small projection head is intentionally used to reduce overparameterization under small-batch CPU constraints and to mitigate representation collapse risk, which can occur when the projection space is excessively high-dimensional relative to dataset size. This design follows empirical findings from SimCLR that the projection head should be sufficient for contrastive alignment while preserving compact backbone representations.

- Optional GPU Projection Head:
  - 128 $\to$ 128 $\to$ 64
  - Larger batch size (128+)
  - Temperature tuning

#### 4. Predictive Coding (Future Embedding Prediction)

- Setup:
  - Given embedding $z_t$, predict $z_{t+1}$.
- Loss:
  - MSE in latent space
- Projection head:
  - Projection head not required.
- Notes:
  - Very light computationally.

---

## Sequence Modeling Architectures

We now evaluate structured sequence prediction.

### A. Feedforward Policy

- Architecture:
  - CNN $\to$ MLP $\to$ action logits
- Memory:
  - No memory.

### B. Recurrent Policy

- Architecture:
  - CNN $\to$ LSTM $\to$ MLP $\to$ action logits
- CPU Configuration:
  - LSTM hidden size: 64
- Optional GPU:
  - Hidden size: 128-256
  - 2 LSTM layers

---

## Evaluation Protocol

To ensure rigorous grading alignment:

1. Next-action prediction accuracy
2. Full trajectory accuracy
3. Sample efficiency (episodes to 80% performance)
4. Generalization to unseen maps
5. Linear probe accuracy on embeddings
6. Embedding separability (cosine similarity distribution)
7. Training time (seconds)
8. Memory footprint (MB)

This directly supports grading criteria for thorough experimentation.

---

## Experimental Design

Must vary:

- CNN depth (3 vs 5 layers)
- Optimizer (Adam vs SGD)
- Learning rate (1e-3 vs 5e-4)
- Embedding dimension (64 vs 128)
- LSTM hidden size (64 vs 128)

All results reported with:

- Test accuracy
- Training time
- Memory usage

### EXPERIMENT MATRIX

#### CPU Configuration Matrix

| Encoder Type | Pretrain | Memory | Optimizer | Emb Dim | LSTM Dim |
| --- | --- | --- | --- | --- | --- |
| Supervised | No | No | Adam | 64 | - |
| Supervised | No | LSTM | Adam | 64 | 64 |
| Contrastive | Yes | No | Adam | 64 | - |
| Contrastive | Yes | LSTM | Adam | 64 | 64 |
| Contrastive | Yes | LSTM | SGD | 64 | 64 |
| Predictive | Yes | LSTM | Adam | 64 | 64 |

#### Optional GPU Extended Matrix

| Encoder Type | Pretrain | Memory | Optimizer | Emb Dim | LSTM Dim | Depth |
| --- | --- | --- | --- | --- | --- | --- |
| Contrastive | Yes | LSTM | Adam | 256 | 128 | 5 |
| Supervised | No | LSTM | Adam | 256 | 128 | 5 |
| Predictive | Yes | LSTM | Adam | 256 | 128 | 5 |





